# Train Machine Learning Models

## Overview
- Test a range of different models.
- Test a range of different hyperparameters.
- Test a range of different inputs. 

In [1]:
# Import libraries.
import os
import random
import json
import pandas as pd
import numpy as np
from glob import glob

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, make_scorer, confusion_matrix
)

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

import pandas as pd
import numpy as np
import random
import torch
import re
import nltk
from nltk.corpus import stopwords

import shap
from itertools import product


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report


from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

import numpy as np
import pandas as pd
import os
import random
from sklearn.decomposition import PCA, FastICA, FactorAnalysis
from sklearn.random_projection import GaussianRandomProjection
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import umap
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import sparse

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

N_SPLITS_CV = 5
N_JOBS = 1  # prioritizing no leakage

In [3]:
# Make stop words.
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))
stop_words.extend(['mrs', 'ms', 'mr', 'am', 'pm'])
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}
# Load the dataset
dataset = pd.read_csv("./dataSyntheticAll.csv")
dataset["label"] = dataset["needs"].map(label_map)

# Make function to remove punctuation, make lowercase, remove names, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        # lowercase + extract words only
        tokens = re.findall(r"\b[a-zA-Z]+\b", note.lower())
        
        # filter stopwords
        tokens = [t for t in tokens if t not in stop_words]
        
        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

dataset['report'] = preprocessing(dataset['report'].values.tolist())

# Split the dataset. (80/10/10)
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=dataset["label"]
)
y_train = train_df['label'].values
y_test = test_df['label'].values

# Check distributions.
def check_distribution(dataframe, name):
    counts = dataframe["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_df, "Train")
check_distribution(test_df, "Test")

[nltk_data] Downloading package stopwords to /home/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Test distribution:
label
0    0.502161
1    0.497839
Name: proportion, dtype: float64


In [4]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # fast and strong
train_embeddings = embedding_model.encode(train_df['report'].tolist(), show_progress_bar=True)
test_embeddings  = embedding_model.encode(test_df['report'].tolist(), show_progress_bar=True)

# tfidf and reduction parameters
tfidf_params = {
    "max_features": [10000, 20000],
    "ngram_range": [(1,1), (1,2)],
    "min_df": [2,5]
}

reduction_methods = {
    "pca": PCA,
    "ica": FastICA,
    "rp": GaussianRandomProjection,
    "fa": FactorAnalysis,
    "umap": umap.UMAP
}

inputs_dict = {'embeddings': train_embeddings}
test_dict = {'embeddings': test_embeddings}

# tf-idf representations
for max_feat, n_gram, min_df in product(tfidf_params['max_features'],
                                       tfidf_params['ngram_range'],
                                       tfidf_params['min_df']):
    key = f"tfidf_{max_feat}_{max(list(n_gram))}_{min_df}"
    vectorizer = TfidfVectorizer(max_features=max_feat, ngram_range=n_gram, min_df=min_df)
    X_train_tfidf = vectorizer.fit_transform(train_df['report'].tolist())
    X_test_tfidf  = vectorizer.transform(test_df['report'].tolist())
    inputs_dict[key] = X_train_tfidf
    test_dict[key] = X_test_tfidf

# embeddings and dimensionality reduction
for name, method in reduction_methods.items():
    for n_components in [10,50,100,200]:
        key = f"{name}_{n_components}"
        if name == "umap":
            reducer = method(n_components=n_components, random_state=RANDOM_STATE, n_neighbors=10)
        else:
            reducer = method(n_components=n_components, random_state=RANDOM_STATE)
        X_train_red = reducer.fit_transform(train_embeddings)
        X_test_red  = reducer.transform(test_embeddings)
        inputs_dict[key] = X_train_red
        test_dict[key] = X_test_red


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/145 [00:00<?, ?it/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  war

In [5]:
# get models - broad coverage across linear, margin, instance-based, and nonlinear tree ensembles
def build_models(random_state):
    models = {
        # high numbers of iterations (max_iter) avoids convergence warnings in high-dimensional spaces
        "LogisticRegression": LogisticRegression(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "RidgeClassifier": RidgeClassifier(class_weight="balanced", random_state=random_state), # strong baseline
        "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "SVC-RBF": SVC(probability=False, random_state=random_state, class_weight="balanced"), # probability set to false saves on compute
        "SGD-Hinge": SGDClassifier(loss="hinge", max_iter=20000, random_state=random_state, class_weight="balanced"), # hinge loss is good for binary classification problems - fast on large data
        "KNN": KNeighborsClassifier(), # deterministic - sensitive to scaling (good to test with and without standard scaler)
        "DecisionTree": DecisionTreeClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "RandomForest": RandomForestClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "ExtraTrees": ExtraTreesClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "GradientBoosting": GradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "HistGB": HistGradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "AdaBoost": AdaBoostClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "GaussianNaiveBayes": GaussianNB(), # deterministic - interpretable
    }
    return models

# get parameter grid to test
def small_param_grid(name): # high-leverage parameter sweeps (runtime will be effected if we test everything)
    grids = {
        # lowered c values due to convergence issues
        "LogisticRegression": {"clf__C": [0.1, 0.5, 1.0]},
        "LinearSVC":          {"clf__C": [0.1, 0.5, 1.0]},
        "SVC-RBF":            {"clf__C": [0.1, 0.5, 1.0], "clf__gamma": ["scale", "auto"]},
        "KNN":                {"clf__n_neighbors": [3, 5, 11]},
        "RandomForest":       {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "ExtraTrees":         {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "HistGB":             {"clf__max_depth": [None, 6, 10]},
    }
    return grids.get(name, None)

# make pipeline with and without scaling
def preprocessor(scale):
    # median ensures it is robust to outliers
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        # scales data to benefit models such as SVM and KNN
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        steps.append(("scaler", FunctionTransformer(lambda X: X, feature_names_out="one-to-one")))
    return Pipeline(steps)


def gmean_binary_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0   # sensitivity
    tnr = tn / (tn + fp) if (tn + fp) else 0.0   # specificity
    return np.sqrt(tpr * tnr)

gmean_scorer = make_scorer(gmean_binary_score)

def build_scorers():
    return {
        "accuracy": "accuracy", # normal accuracy
        "balanced_accuracy": make_scorer(balanced_accuracy_score), # balanced accuracy
        "precision": make_scorer(precision_score, average="binary", zero_division=0),
        "recall": make_scorer(recall_score, average="binary", zero_division=0),
        "f1": make_scorer(f1_score, average="binary", zero_division=0),
        "roc_auc": "roc_auc",
        "gmean": gmean_scorer
    }


In [6]:
def evaluate_one(X_train, y_train, X_test, y_test, feature_name, variant,
                 random_state=RANDOM_STATE, n_splits=N_SPLITS_CV, n_jobs=N_JOBS):

    inner_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scorers = build_scorers()
    results = []

    for name, clf in build_models(random_state).items():
        # scaling decision
        X_tr, X_te = X_train, X_test

        # Convert sparse to dense for models that need dense arrays
        if sparse.issparse(X_tr):
            if name in ["HistGB", "GradientBoosting", "GaussianNaiveBayes"]:
                X_tr = X_tr.toarray()
                X_te = X_te.toarray()

        # Scaling decision
        if variant == "scaled":
            scaler = StandardScaler(with_mean=False if sparse.issparse(X_tr) else True)
        else:
            scaler = FunctionTransformer(lambda X: X, feature_names_out="one-to-one")

        pipe = Pipeline([
            ("scale", scaler),
            ("clf", clf)
        ])

        grid = small_param_grid(name)
        model_for_cv = GridSearchCV(pipe, grid, scoring=scorers, cv=inner_cv,
                                    n_jobs=n_jobs, refit="gmean") if grid else pipe

        cv_out = cross_validate(model_for_cv, X_tr, y_train, cv=outer_cv,
                                scoring=scorers, return_train_score=False,
                                n_jobs=n_jobs, return_estimator=True)

        # Fit on full training
        model_for_cv.fit(X_tr, y_train)
        y_pred = model_for_cv.predict(X_te)

        # best estimator info
        if isinstance(model_for_cv, GridSearchCV):
            chosen = model_for_cv.best_estimator_
            tuned = model_for_cv.best_params_
        else:
            chosen = model_for_cv
            tuned = {}

        clf_params = chosen.named_steps["clf"].get_params()
        tuned_json = json.dumps(tuned, default=str)
        clf_json = json.dumps(clf_params, default=str)

        y_score = None
        if hasattr(model_for_cv, "predict_proba"):
            y_score = model_for_cv.predict_proba(X_te)[:, 1]
        elif hasattr(model_for_cv, "decision_function"):
            y_score = model_for_cv.decision_function(X_te)

        row = {
            "feature_set": feature_name,
            "variant": variant,
            "model": name,
            "tuned_parameters": tuned_json,
            "clf_parameters": clf_json,
            **{f"cv_{k.replace('test_','')}_mean": float(np.mean(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            **{f"cv_{k.replace('test_','')}_std": float(np.std(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            "holdout_accuracy": accuracy_score(y_test, y_pred),
            "holdout_bal_acc": balanced_accuracy_score(y_test, y_pred),
            "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
            "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
            "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
            "holdout_roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan,
            "holdout_gmean": gmean_binary_score(y_test, y_pred)
        }

        results.append(row)
        print(row)
    return pd.DataFrame(results)

In [ ]:
all_results = []
for feature_name, X_train in inputs_dict.items():
    X_test = test_dict[feature_name]

    for variant in ("raw", "scaled"):
        df_res = evaluate_one(X_train, y_train, X_test, y_test, feature_name, variant)
        all_results.append(df_res)

# Concatenate all results and save.
final_results = pd.concat(all_results, ignore_index=True)
final_results.to_csv("baseline_model_results.csv", index=False)

{'feature_set': 'embeddings', 'variant': 'raw', 'model': 'LogisticRegression', 'tuned_parameters': '{"clf__C": 1.0}', 'clf_parameters': '{"C": 1.0, "class_weight": "balanced", "dual": false, "fit_intercept": true, "intercept_scaling": 1, "l1_ratio": 0.0, "max_iter": 20000, "n_jobs": null, "penalty": "deprecated", "random_state": 1618, "solver": "lbfgs", "tol": 0.0001, "verbose": 0, "warm_start": false}', 'cv_accuracy_mean': 0.8843502422508902, 'cv_balanced_accuracy_mean': 0.884454534418467, 'cv_precision_mean': 0.8668893310311064, 'cv_recall_mean': 0.9070866735829484, 'cv_f1_mean': 0.88648661428689, 'cv_roc_auc_mean': 0.9501831986532798, 'cv_gmean_mean': 0.8841344402582546, 'cv_accuracy_std': 0.009487095159951484, 'cv_balanced_accuracy_std': 0.009490637504293645, 'cv_precision_std': 0.010975020614202906, 'cv_recall_std': 0.011618466269500902, 'cv_f1_std': 0.009260099180156073, 'cv_roc_auc_std': 0.0031152116620973406, 'cv_gmean_std': 0.009485911558573298, 'holdout_accuracy': 0.878133102

In [ ]:
final_results